# Scratchpad notebook
To do "back of the page" calculations, manipulations, changed, etc.

In [12]:
import pandas as pd
from pathlib import Path

# -------------------------
# Configuration
# -------------------------
root_dir = Path.cwd().parent
data_dir = root_dir / "data"
prompts_dir = data_dir / "gender_prompts"
results_dir = data_dir / "olmo7b_results" / "v2"

# I want to load the "extracted_*assumed*.json" files, get their gender_assigned column, 
# and append it to the regular "olmo*assumed*.json" files for smoother analysis

models = ["base", "sft", "dpo", "rlvr"]

def load_assumed_files(results_dir: Path = results_dir) -> dict:
    """Load all assumed output files from the results directory and return as Dataframe in dict"""
    # Get all assumed files from the results dir
    all_assumed_files = list(results_dir.glob("*assumed*.json"))
    print("Found the following files in results_dir:")
    for file in all_assumed_files:
        print(file.name)
    # Make dict with each file as a df and then convert to Dataset for HF pipeline
    all_assumed_dfs = {file.name: pd.read_json(file) for file in all_assumed_files}

    return all_assumed_dfs

def swap_gender_col(model, extracted_df, regular_df):
    print(f"Swapping for [{model}]...")
    gender = extracted_df["assigned_gender"].tolist()
    regular_df["gender"] = gender

In [2]:
assumed_dfs = load_assumed_files()

Found the following files in results_dir:
extracted_olmo7b_base_assumed_all_temps.json
extracted_olmo7b_dpo_assumed_all_temps.json
extracted_olmo7b_rlvr_assumed_all_temps.json
extracted_olmo7b_sft_assumed_all_temps.json
olmo7b_base_assumed_all_temps.json
olmo7b_dpo_assumed_all_temps.json
olmo7b_rlvr_assumed_all_temps.json
olmo7b_sft_assumed_all_temps.json


In [13]:
for model in models:
    dfs = []                    # go through one model category at a time
    for k, v in assumed_dfs.items():    # go through all the jsons
        if model in k:                  # get the jsons for that model
            print(f"[{model}], {k}")
            dfs.append(v)
            print(len(dfs))
        if len(dfs) == 2:    
            swap_gender_col(model, dfs[0], dfs[1])    

[base], extracted_olmo7b_base_assumed_all_temps.json
1
[base], olmo7b_base_assumed_all_temps.json
2
Swapping for [base]...
Swapping for [base]...
Swapping for [base]...
Swapping for [base]...
[sft], extracted_olmo7b_sft_assumed_all_temps.json
1
[sft], olmo7b_sft_assumed_all_temps.json
2
Swapping for [sft]...
[dpo], extracted_olmo7b_dpo_assumed_all_temps.json
1
[dpo], olmo7b_dpo_assumed_all_temps.json
2
Swapping for [dpo]...
Swapping for [dpo]...
Swapping for [dpo]...
[rlvr], extracted_olmo7b_rlvr_assumed_all_temps.json
1
[rlvr], olmo7b_rlvr_assumed_all_temps.json
2
Swapping for [rlvr]...
Swapping for [rlvr]...


In [15]:
for k, v in assumed_dfs.items():
    if "extracted" not in k:
        print(f"{k}: {v.columns}")
        v.to_json(results_dir / k, orient="records", indent=4)

olmo7b_base_assumed_all_temps.json: Index(['model_key', 'model_name', 'profile_id', 'temperature',
       'occupation_category', 'attended_university', 'response_number',
       'response', 'mean_entropy', 'max_entropy', 'min_entropy', 'std_entropy',
       'mean_entropy_nucleus', 'max_entropy_nucleus', 'min_entropy_nucleus',
       'std_entropy_nucleus', 'gender'],
      dtype='object')
olmo7b_dpo_assumed_all_temps.json: Index(['model_key', 'model_name', 'profile_id', 'temperature',
       'occupation_category', 'attended_university', 'response_number',
       'response', 'mean_entropy', 'max_entropy', 'min_entropy', 'std_entropy',
       'mean_entropy_nucleus', 'max_entropy_nucleus', 'min_entropy_nucleus',
       'std_entropy_nucleus', 'gender'],
      dtype='object')
olmo7b_rlvr_assumed_all_temps.json: Index(['model_key', 'model_name', 'profile_id', 'temperature',
       'occupation_category', 'attended_university', 'response_number',
       'response', 'mean_entropy', 'max_entropy'